In [0]:
from pyspark.sql.functions import col, expr, substring

# 1. Read the CSV and analyze structure
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day", IntegerType(), True),
    StructField("hour", StringType(), True),  # Force as STRING
    StructField("temperature_2m (°C)", DoubleType(), True),
    StructField("apparent_temperature (°C)", DoubleType(), True)
])

weather_df = spark.read.csv(
    "/Volumes/workspace/default/dbfs/Toronto_weather_data.csv",
    header=True,
    schema=schema,
    sep=";"
)

print(f"Number of rows: {weather_df.count()}")
print(f"Number of columns: {len(weather_df.columns)}")

print("Sample of loaded data:")
weather_df.show(5, truncate=False)

# 2. Clean column names
weather_df = weather_df \
    .withColumnRenamed("temperature_2m (°C)", "temperature_2m_celsius") \
    .withColumnRenamed("apparent_temperature (°C)", "apparent_temperature_celsius")

# 3. Transform hour - use substring to get only first 5 characters (HH:MM)
weather_transformed = weather_df.withColumn(
    "hour",
    substring(col("hour"), 1, 5)  # Take only first 5 characters: "00:00"
)

print("Sample of transformed data:")
weather_transformed.select("year", "month", "day", "hour", "temperature_2m_celsius", "apparent_temperature_celsius").show(10, truncate=False)

# 4. Save the raw (bronze) data as parquet
raw_output_path = "/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather/"
weather_df.write.mode("overwrite").parquet(raw_output_path)
print(f"Raw parquet saved to: {raw_output_path}")

# 5. Save the transformed (silver) data as parquet
silver_output_path = "/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly/"
weather_transformed.write.mode("overwrite").parquet(silver_output_path)
print(f"Transformed parquet saved to: {silver_output_path}")

# 6. Verify the files were created
print("\n📁 Raw parquet files:")
display(dbutils.fs.ls(raw_output_path))

print("\n📁 Silver parquet files:")
display(dbutils.fs.ls(silver_output_path))

# 7. Show final result
print("Final transformed data sample:")
weather_transformed.show(20, truncate=False)

Number of rows: 17544
Number of columns: 6
Sample of loaded data:
+----+-----+---+----+-------------------+-------------------------+
|year|month|day|hour|temperature_2m (°C)|apparent_temperature (°C)|
+----+-----+---+----+-------------------+-------------------------+
|2022|10   |1  |0:00|12.3               |10.6                     |
|2022|10   |1  |1:00|12.6               |11.3                     |
|2022|10   |1  |2:00|11.6               |10.2                     |
|2022|10   |1  |3:00|10.8               |9.1                      |
|2022|10   |1  |4:00|9.9                |8.2                      |
+----+-----+---+----+-------------------+-------------------------+
only showing top 5 rows
Sample of transformed data:
+----+-----+---+----+----------------------+----------------------------+
|year|month|day|hour|temperature_2m_celsius|apparent_temperature_celsius|
+----+-----+---+----+----------------------+----------------------------+
|2022|10   |1  |0:00|12.3                  |10.6

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather/_SUCCESS,_SUCCESS,0,1770872178000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather/_committed_1426476166793799561,_committed_1426476166793799561,124,1770871221000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather/_committed_2643941214379450562,_committed_2643941214379450562,234,1770871380000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather/_committed_623758070534473330,_committed_623758070534473330,222,1770871556000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather/_committed_6261527490891764077,_committed_6261527490891764077,222,1770872053000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather/_committed_6943825120420731712,_committed_6943825120420731712,223,1770872178000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather/_started_1426476166793799561,_started_1426476166793799561,0,1770871221000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather/_started_2643941214379450562,_started_2643941214379450562,0,1770871380000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather/_started_623758070534473330,_started_623758070534473330,0,1770871556000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather/_started_6261527490891764077,_started_6261527490891764077,0,1770872053000



📁 Silver parquet files:


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly/_SUCCESS,_SUCCESS,0,1770872180000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly/_committed_1740325068951930234,_committed_1740325068951930234,223,1770871558000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly/_committed_4348734900273361429,_committed_4348734900273361429,234,1770871381000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly/_committed_6483411748643068163,_committed_6483411748643068163,124,1770871223000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly/_committed_8816131674244524358,_committed_8816131674244524358,222,1770872180000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly/_committed_922447292341319915,_committed_922447292341319915,222,1770872055000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly/_started_1740325068951930234,_started_1740325068951930234,0,1770871558000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly/_started_4348734900273361429,_started_4348734900273361429,0,1770871381000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly/_started_6483411748643068163,_started_6483411748643068163,0,1770871222000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly/_started_8816131674244524358,_started_8816131674244524358,0,1770872179000


Final transformed data sample:
+----+-----+---+-----+----------------------+----------------------------+
|year|month|day|hour |temperature_2m_celsius|apparent_temperature_celsius|
+----+-----+---+-----+----------------------+----------------------------+
|2022|10   |1  |0:00 |12.3                  |10.6                        |
|2022|10   |1  |1:00 |12.6                  |11.3                        |
|2022|10   |1  |2:00 |11.6                  |10.2                        |
|2022|10   |1  |3:00 |10.8                  |9.1                         |
|2022|10   |1  |4:00 |9.9                   |8.2                         |
|2022|10   |1  |5:00 |9.7                   |7.6                         |
|2022|10   |1  |6:00 |9.6                   |7.7                         |
|2022|10   |1  |7:00 |9.5                   |7.5                         |
|2022|10   |1  |8:00 |10.1                  |7.8                         |
|2022|10   |1  |9:00 |10.4                  |7.9                     